# LayerNorm 手动实现与公式讲解

这个 notebook 用 Transformer 中最常见的张量形状来讲解 LayerNorm：

- 输入隐藏状态：`x.shape = [batch_size, seq_len, hidden_dim]`
- LayerNorm 归一化维度：最后一维 `hidden_dim`
- 每个 token 单独计算自己的均值和方差
- 不调用 `torch.nn.LayerNorm` 实现核心逻辑，先完全手动写一遍，再和 PyTorch 官方实现做数值对齐

在本项目的 `model.py` 中，`MultiHeadAttention` 和 `Mlp` 里都有类似写法：

```python
self.layer_norm = nn.LayerNorm(embeding_size)
```

这里的 `embeding_size` 就对应下面公式里的 `D = hidden_dim`。

## 1. LayerNorm 的核心公式

假设输入张量：

$$x \in \mathbb{R}^{B \times T \times D}$$

其中：

- `B` = batch size
- `T` = sequence length
- `D` = hidden dimension，也就是 embedding 维度

LayerNorm 对每个样本、每个 token 的最后一维单独做归一化。也就是说，对固定的 `(b, t)`，只在 `D` 这一维上统计均值和方差。

均值公式：

$$\mu_{b,t} = \frac{1}{D}\sum_{i=1}^{D} x_{b,t,i}$$

方差公式：

$$\sigma^2_{b,t} = \frac{1}{D}\sum_{i=1}^{D}(x_{b,t,i} - \mu_{b,t})^2$$

归一化公式：

$$\hat{x}_{b,t,i} = \frac{x_{b,t,i} - \mu_{b,t}}{\sqrt{\sigma^2_{b,t} + \epsilon}}$$

带可学习参数的输出公式：

$$y_{b,t,i} = \gamma_i \hat{x}_{b,t,i} + \beta_i$$

其中：

- `epsilon` 防止除以 0
- `gamma.shape = [D]`，缩放参数
- `beta.shape = [D]`，平移参数
- PyTorch 的 `nn.LayerNorm(D)` 默认会学习 `gamma` 和 `beta`

注意：这里的方差除以 `D`，不是除以 `D - 1`。也就是 PyTorch 里 `unbiased=False` 的方差。

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

print("torch version:", torch.__version__)

## 2. 构造一个 Transformer 风格的输入

下面用一个小张量模拟 Transformer 中某一层的 hidden states：

$$x.shape = [B, T, D] = [2, 3, 4]$$

含义是：

- `B = 2`：一个 batch 中有 2 个句子
- `T = 3`：每个句子有 3 个 token
- `D = 4`：每个 token 的隐藏向量维度是 4

LayerNorm 会对每个 token 的 4 维向量单独做归一化。

In [ ]:
x = torch.tensor(
    [
        [
            [1.0, 2.0, 3.0, 4.0],
            [2.0, 4.0, 6.0, 8.0],
            [1.0, 1.0, 1.0, 1.0],
        ],
        [
            [10.0, 20.0, 30.0, 40.0],
            [-1.0, 0.0, 1.0, 2.0],
            [3.0, 5.0, 7.0, 9.0],
        ],
    ],
    dtype=torch.float32,
)

print("x.shape:", x.shape)
print(x)

## 3. 完全手动实现 LayerNorm 函数

下面这个函数没有调用 `nn.LayerNorm`，只用基础张量操作实现：

1. `mean = x.mean(dim=-1, keepdim=True)`
2. `var = ((x - mean) ** 2).mean(dim=-1, keepdim=True)`
3. `x_hat = (x - mean) / sqrt(var + eps)`
4. `y = gamma * x_hat + beta`

为什么 `keepdim=True`：

- `x.shape = [B, T, D]`
- `mean.shape = [B, T, 1]`
- 这样 `x - mean` 时可以自动广播回 `[B, T, D]`

In [ ]:
def layer_norm_manual(x, gamma=None, beta=None, eps=1e-5):
    """完全手动实现 LayerNorm，不调用 torch.nn.LayerNorm。

    参数：
        x: 输入张量，常见形状是 [B, T, D]
        gamma: 缩放参数，形状 [D]
        beta: 平移参数，形状 [D]
        eps: 防止除以 0 的小常数

    返回：
        y: LayerNorm 输出，形状和 x 相同
        mean: 最后一维均值，形状 [B, T, 1]
        var: 最后一维方差，形状 [B, T, 1]
        x_hat: 标准化后的张量，形状和 x 相同
    """
    hidden_dim = x.size(-1)

    if gamma is None:
        gamma = torch.ones(hidden_dim, dtype=x.dtype, device=x.device)
    if beta is None:
        beta = torch.zeros(hidden_dim, dtype=x.dtype, device=x.device)

    # 对最后一维 D 计算均值。
    # x:    [B, T, D]
    # mean: [B, T, 1]
    mean = x.mean(dim=-1, keepdim=True)

    # 对最后一维 D 计算方差。
    # 这里除以 D，对应 PyTorch LayerNorm 的 unbiased=False。
    # var: [B, T, 1]
    var = ((x - mean) ** 2).mean(dim=-1, keepdim=True)

    # 标准化：每个 token 的 D 维向量变成均值约为 0、方差约为 1。
    # x_hat: [B, T, D]
    x_hat = (x - mean) / torch.sqrt(var + eps)

    # gamma 和 beta 的形状是 [D]，会广播到 [B, T, D]。
    # y: [B, T, D]
    y = x_hat * gamma + beta
    return y, mean, var, x_hat

## 4. 查看中间结果的形状

这里先让 `gamma = 1`、`beta = 0`，也就是只看纯标准化效果。

如果某个 token 的向量所有值都一样，比如 `[1, 1, 1, 1]`，它的方差是 0。加上 `eps` 后不会除以 0，输出会变成全 0。

In [ ]:
manual_y, mean, var, x_hat = layer_norm_manual(x)

print("x.shape:       ", x.shape)
print("mean.shape:    ", mean.shape)
print("var.shape:     ", var.shape)
print("x_hat.shape:   ", x_hat.shape)
print("manual_y.shape:", manual_y.shape)

print("\nmean:")
print(mean.squeeze(-1))

print("\nvar:")
print(var.squeeze(-1))

print("\nmanual_y:")
print(manual_y)

## 5. 验证归一化后的均值和方差

标准化后的 `x_hat` 理论上满足：

$$mean(\hat{x}_{b,t,:}) \approx 0$$

$$var(\hat{x}_{b,t,:}) \approx 1$$

如果某个 token 原始方差为 0，那么输出会是全 0，它的方差仍然是 0。这是正常现象。

In [ ]:
print("x_hat mean over hidden_dim:")
print(x_hat.mean(dim=-1))

print("\nx_hat var over hidden_dim, unbiased=False:")
print(x_hat.var(dim=-1, unbiased=False))

## 6. 加上可学习参数 gamma 和 beta

LayerNorm 不只是标准化，它还有两个可学习参数：

$$y_{b,t,i} = \gamma_i \hat{x}_{b,t,i} + \beta_i$$

直观理解：

- `gamma` 控制每个 hidden 维度的缩放
- `beta` 控制每个 hidden 维度的平移
- 归一化先把分布拉到稳定范围，再让模型自己学是否要放大、缩小或移动某些维度

In [ ]:
gamma = torch.tensor([1.0, 1.5, 0.5, 2.0])
beta = torch.tensor([0.0, 0.1, -0.2, 0.3])

manual_y_affine, mean, var, x_hat = layer_norm_manual(x, gamma=gamma, beta=beta)

print("gamma.shape:", gamma.shape)
print("beta.shape: ", beta.shape)
print("manual_y_affine.shape:", manual_y_affine.shape)
print(manual_y_affine)

## 7. 和 PyTorch `nn.LayerNorm` 对齐验证

下面只把 PyTorch 的 `nn.LayerNorm` 当作校验器，不把它当作实现方式。

对齐条件：

- `normalized_shape = D`
- `eps` 一致
- `weight` 复制为手动实现里的 `gamma`
- `bias` 复制为手动实现里的 `beta`

如果手写公式正确，两个输出的最大误差应该非常接近 0。

In [ ]:
torch_ln = nn.LayerNorm(normalized_shape=x.size(-1), eps=1e-5)

with torch.no_grad():
    torch_ln.weight.copy_(gamma)
    torch_ln.bias.copy_(beta)

torch_y = torch_ln(x)
max_abs_diff = (manual_y_affine - torch_y).abs().max()

print("torch_y.shape:", torch_y.shape)
print("max_abs_diff:", max_abs_diff.item())
print("allclose:", torch.allclose(manual_y_affine, torch_y, atol=1e-6))

## 8. 完全手写一个可训练的 LayerNorm 模块

下面实现一个 `ManualLayerNorm`：

- 不调用 `nn.LayerNorm`
- 只把 `gamma` 和 `beta` 包装成 `nn.Parameter`
- 前向传播仍然按公式手动计算

这就等价于自己写了一个可以放进模型训练的 LayerNorm 层。

In [ ]:
class ManualLayerNorm(nn.Module):
    """手写可训练 LayerNorm。

    normalized_shape 对应最后一维 D。
    对 Transformer hidden states 来说，输入通常是 [B, T, D]。
    """

    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()
        self.normalized_shape = normalized_shape
        self.eps = eps

        # gamma 和 beta 是可学习参数，形状都是 [D]。
        self.gamma = nn.Parameter(torch.ones(normalized_shape))
        self.beta = nn.Parameter(torch.zeros(normalized_shape))

    def forward(self, x):
        # x: [B, T, D]
        mean = x.mean(dim=-1, keepdim=True)                 # [B, T, 1]
        var = ((x - mean) ** 2).mean(dim=-1, keepdim=True)  # [B, T, 1]
        x_hat = (x - mean) / torch.sqrt(var + self.eps)     # [B, T, D]
        return x_hat * self.gamma + self.beta               # [B, T, D]

## 9. 用 Transformer 形状测试手写模块

这里模拟一层 Transformer 的输出：

$$hidden.shape = [B, T, D] = [2, 5, 8]$$

LayerNorm 后形状不变，仍然是 `[2, 5, 8]`。它只改变数值分布，不改变张量尺寸。

In [ ]:
B, T, D = 2, 5, 8
hidden = torch.randn(B, T, D) * 3 + 10

manual_ln_module = ManualLayerNorm(D)
out = manual_ln_module(hidden)

print("hidden.shape:", hidden.shape)
print("out.shape:   ", out.shape)

print("\nBefore LayerNorm, mean over D:")
print(hidden.mean(dim=-1))

print("\nAfter LayerNorm, mean over D:")
print(out.mean(dim=-1))

print("\nAfter LayerNorm, var over D, unbiased=False:")
print(out.var(dim=-1, unbiased=False))

## 10. 验证手写模块可以反向传播

因为实现里只用了 PyTorch 的基础张量运算，autograd 可以自动求导。

这说明手写 LayerNorm 可以像普通层一样参与训练。

In [ ]:
loss = (out ** 2).mean()
loss.backward()

print("loss:", loss.item())
print("gamma.grad.shape:", manual_ln_module.gamma.grad.shape)
print("beta.grad.shape: ", manual_ln_module.beta.grad.shape)
print("gamma.grad:", manual_ln_module.gamma.grad)
print("beta.grad: ", manual_ln_module.beta.grad)

## 11. 和 Transformer 残差结构放在一起理解

本项目 `model.py` 里的注意力和 MLP 都用了残差连接再 LayerNorm，例如：

```python
return self.layer_norm(output + q), attn
```

抽象成公式就是：

$$z = x + Sublayer(x)$$

$$LayerNorm(z) = \gamma \cdot \frac{z - mean(z)}{\sqrt{var(z) + \epsilon}} + \beta$$

这里的 `mean(z)` 和 `var(z)` 仍然只沿最后一维 `D` 计算。

直观作用：

- 残差连接保留原始信息，缓解深层网络训练困难
- LayerNorm 稳定每个 token 的 hidden 向量分布
- Transformer 中 batch 内句子长度不同、padding 不同，但 LayerNorm 不依赖 batch 统计，所以比 BatchNorm 更适合 NLP 序列建模

## 12. 小结

LayerNorm 的关键点：

1. 对最后一维做归一化，Transformer 里通常就是 `hidden_dim / embeding_size`。
2. 每个 token 独立统计均值和方差，不跨 batch，不跨 seq_len。
3. 输出形状和输入形状完全相同。
4. `gamma` 和 `beta` 是可学习参数，形状是 `[hidden_dim]`。
5. PyTorch 的方差计算对应 `unbiased=False`，也就是除以 `D`。
6. 手写实现只需要 `mean`、平方差均值、`sqrt`、广播和两个可学习参数。